# Ejercicios — Regresión Lineal introducción, cuadrados mínimos, descenso por gradiente

- 📘 Explicación: [`../explained/2_regresion_lineal_cuadrados_minimos.md`](../explained/2_regresion_lineal_cuadrados_minimos.md)
- 📓 Notebook de clase: [`../raw/2_regresion_lineal_cuadrados_minimos.ipynb`](../raw/2_regresion_lineal_cuadrados_minimos.ipynb)

> Los enunciados están tal cual los dio la cátedra. Las pistas (*centros*) están al final del `.md` explicado.


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../../datasets")


---

### Ejercicio 1 — Mínimo cuadrático vs. mínimo absoluto de un conjunto de valores

Dado un conjunto de valores $x_1, x_2, ..., x_N$
1. ¿Qué valor $x$ minimiza el error cuadrático ($\sum_{i=1}^{N} (x_i - x)^2$)?
1. ¿Y si el error es absoluto ($\sum_{i=1}^{N} |x_i - x|$)?


**Mi resolución:**

<!-- 1.1 — ¿Qué valor $x$ minimiza el error cuadrático ($\sum_{i=1}^{N} (x_i - x)^2$)? -->


<!-- 1.2 — ¿Y si el error es absoluto ($\sum_{i=1}^{N} |x_i - x|$)? -->


---

### Ejercicio 2 — La ecuación normal a mano, con una sola variable

Considerar el caso para una única variable predictora $X_1$:
1. Escribir la función de costo $RSS(\beta)$ sin usar la notación matricial. ¿Cuántas componentes tiene $\beta$?
1. Expandir la expresión y mostrar que es una función cuadrática en $\beta$.
1. Encontrar el mínimo de la función cuadrática y mostrar que se obtiene la solución de mínimos cuadrados:

$$ \hat{\beta_1} = \frac{\sum_{i=1}^{N} (x_i - \bar{x})(y_i - \bar{y})}{\sum_{i=1}^{N} (x_i - \bar{x})^2}$$

$$ \hat{\beta_0} = \bar{y} - \hat{\beta_1} \bar{x}$$

1. ¿Cómo se modifican esas expresiones si consideramos predictores estandarizados?¿Y si la variable de respuesta también está estandarizada?
4. Plantear la expresión para el caso de $p = 2$ variables predictoras. ¿Cuántas componentes tiene $\beta$? ¿Cuántas ecuaciones hay que resolver para encontrar la solución de mínimos cuadrados? Reflexionar sobre la ventaja de usar la formulación matricial.


**Mi resolución:**

<!-- 2.1 — Escribir la función de costo $RSS(\beta)$ sin usar la notación matricial -->


<!-- 2.2 — Expandir la expresión y mostrar que es una función cuadrática en $\beta$ -->


<!-- 2.3 — Encontrar el mínimo de la función cuadrática y mostrar que se obtiene la solución de mínimos… -->


<!-- 2.4 — ¿Cómo se modifican esas expresiones si consideramos predictores estandarizados?¿Y si la variable… -->


<!-- 2.5 — Plantear la expresión para el caso de $p = 2$ variables predictoras -->


---

### Ejercicio 3 — Prostate Cancer: replicar tablas 3.1/3.2 y el estadístico F

Utilizando el dataset de *Prostate Cancer*:
1. Reproducir las Tabla 3.1 y 3.2 del libro Elements of Statistical Learning. Utilizar la librería `statsmodels` de Python para ajustar el modelo lineal. Comparar con los resultados obtenidos en el libro.
1. Reproducir la Tabla 3.2 del libro Elements of Statistical Learning utilizando un estimador de cuadrados mínimos hecho exclusivamente con `numpy`. Comparar con los resultados obtenidos en el libro y con los obtenidos con `statsmodels`.
1. Obtener el estadístico $F$ para el modelo completo y el modelo reducido sin las variables $age$, $lcp$, $gleason$ y $pgg45$. Comparar con el valor obtenido en el libro. Interpretar.
1. Obtener el error en el conjunto de evaluación. Obtener el "benchmark" propuesto en el libro. Graficar las predicciones del modelo completo ($y$ vs $y_{pred}$) y del benchmark.


In [ ]:
# 3.1 — Reproducir las Tablas 3.1 y 3.2 del libro (Elements of Statistical Learning, ESL)
import statsmodels.api as sm

# --- Carga del dataset -----------------------------------------------------
# El archivo es TSV. La primera columna es un indice sin nombre -> la usamos como index.
prostate = pd.read_csv(DATA_DIR / "prostate.data", sep="\t", index_col=0)

predictores = ["lcavol", "lweight", "age", "lbph", "svi", "lcp", "gleason", "pgg45"]
respuesta = "lpsa"

# --- Estandarizacion de los predictores ----------------------------------
# El libro estandariza (media 0, desvio 1) las 8 variables predictoras ANTES de
# separar train/test, es decir usando las estadisticas de las 97 observaciones.
# (Si se estandariza solo con el train los coeficientes cambian un poco:
#  lcavol pasa de 0.68 a 0.72, etc. El ajuste -los Z score- es identico.)
prostate_std = prostate.copy()
prostate_std[predictores] = (
    (prostate[predictores] - prostate[predictores].mean()) / prostate[predictores].std()
)

# La columna 'train' marca con 'T'/'F' la particion que usa el libro (67 / 30).
train = prostate_std[prostate_std["train"] == "T"]
test = prostate_std[prostate_std["train"] == "F"]

X_train, y_train = train[predictores], train[respuesta]
X_test, y_test = test[predictores], test[respuesta].to_numpy()
print(f"Entrenamiento: {len(train)} filas | Evaluacion: {len(test)} filas")

# --- TABLA 3.1: correlaciones entre predictores (en el train) ------------
# El libro muestra solo el triangulo inferior, sin la diagonal, redondeado a 3.
corr = X_train.corr()
triangulo_superior = np.triu(np.ones(corr.shape, dtype=bool))   # mascara: diagonal + arriba
tabla_3_1 = corr.mask(triangulo_superior).iloc[1:, :-1].round(3)
print("\n===== TABLA 3.1 - Correlaciones de los predictores =====")
print(tabla_3_1.fillna(""))

# --- TABLA 3.2: ajuste del modelo lineal con statsmodels ----------------
# statsmodels NO agrega la ordenada al origen: hay que sumar una columna de 1s.
X_train_sm = sm.add_constant(X_train)
modelo = sm.OLS(y_train, X_train_sm).fit()

# El "Z score" del libro es coef / error_estandar (lo que statsmodels llama 't').
tabla_3_2 = pd.DataFrame({
    "Coefficient": modelo.params,
    "Std. Error": modelo.bse,
    "Z Score": modelo.tvalues,
}).round(2).rename(index={"const": "Intercept"})
print("\n===== TABLA 3.2 - Modelo lineal (statsmodels) =====")
print(tabla_3_2)


In [ ]:
# 3.2 — La misma Tabla 3.2 pero con un estimador de cuadrados minimos "a mano" (solo numpy)

# Matriz de diseno X: columna de 1s (intercepto) + los 8 predictores estandarizados.
X = np.column_stack([np.ones(len(X_train)), X_train.to_numpy()])
y = y_train.to_numpy()
n, p = X.shape          # n = 67 observaciones ; p = 9 parametros (1 intercepto + 8 predictores)

# Ecuacion normal:  beta_hat = (X^T X)^-1 X^T y
XtX_inv = np.linalg.inv(X.T @ X)
beta = XtX_inv @ X.T @ y

# Varianza del ruido estimada:  sigma^2 = RSS / (n - p)
residuos = y - X @ beta
rss = residuos @ residuos
sigma2 = rss / (n - p)

# Cov(beta_hat) = sigma^2 (X^T X)^-1  ->  el error estandar es la raiz de su diagonal.
errores_std = np.sqrt(np.diag(sigma2 * XtX_inv))
z_scores = beta / errores_std

tabla_3_2_numpy = pd.DataFrame({
    "Coefficient": beta,
    "Std. Error": errores_std,
    "Z Score": z_scores,
}, index=["Intercept"] + predictores).round(2)
print("===== TABLA 3.2 - Modelo lineal (numpy) =====")
print(tabla_3_2_numpy)

# Chequeo: coincide con statsmodels? (deberia, hasta el error de redondeo de la maquina)
print("\nDiferencia maxima de coeficientes vs statsmodels:",
      np.abs(beta - modelo.params.to_numpy()).max())


In [ ]:
# 3.3 — Estadistico F: aportan algo age, lcp, gleason y pgg45?

from scipy import stats

# Modelo completo -> ya ajustado en 3.1 como 'modelo' (8 predictores).
# Modelo reducido -> se quitan age, lcp, gleason y pgg45  (quedan 4 predictores).
predictores_reducido = ["lcavol", "lweight", "lbph", "svi"]
modelo_reducido = sm.OLS(y_train, sm.add_constant(X_train[predictores_reducido])).fit()

rss_completo = modelo.ssr             # suma de cuadrados de los residuos (RSS)
rss_reducido = modelo_reducido.ssr
p_completo = int(modelo.df_model)           # 8  (nro de predictores del modelo completo)
p_reducido = int(modelo_reducido.df_model)  # 4  (nro de predictores del modelo reducido)
N = int(modelo.nobs)                        # 67

# F = [ (RSS_red - RSS_full) / (p_full - p_red) ]  /  [ RSS_full / (N - p_full - 1) ]
gl_num = p_completo - p_reducido      # 4  -> restricciones que impone el modelo reducido
gl_den = N - p_completo - 1           # 58 -> grados de libertad del modelo completo
F = ((rss_reducido - rss_completo) / gl_num) / (rss_completo / gl_den)
p_valor = stats.f.sf(F, gl_num, gl_den)   # P(F_{gl_num, gl_den} > F observado)

print(f"RSS modelo completo : {rss_completo:.3f}")
print(f"RSS modelo reducido : {rss_reducido:.3f}")
print(f"F = {F:.3f}   (grados de libertad: {gl_num}, {gl_den})")
print(f"p-valor = {p_valor:.3f}")

# Verificacion con el metodo que ya trae statsmodels:
print("statsmodels compare_f_test:", modelo.compare_f_test(modelo_reducido))

print(
    "\nInterpretacion: el libro obtiene F = 1.67 (p = 0.17). Como p > 0.05, no hay\n"
    "evidencia para rechazar H0: los 4 coeficientes son simultaneamente cero.\n"
    "Se pueden descartar esas 4 variables sin una perdida significativa de ajuste."
)


In [ ]:
# 3.4 — Error en el conjunto de evaluacion, benchmark del libro y grafico

# El test ya quedo estandarizado con las MISMAS estadisticas que el train (se hizo
# en 3.1, antes de partir el dataset), asi que solo hay que predecir.
y_pred_completo = modelo.predict(sm.add_constant(X_test, has_constant="add")).to_numpy()

# "Benchmark" del libro (Tabla 3.3, fila 'Base error'): predecir SIEMPRE la media
# de lpsa en el entrenamiento, sin mirar ningun predictor.
y_pred_benchmark = np.full_like(y_test, y_train.mean())


def mse(y_real, y_est):
    """Error cuadratico medio."""
    return np.mean((y_real - y_est) ** 2)


def se_mse(y_real, y_est):
    """Error estandar de ese promedio (como reporta la Tabla 3.3 del libro)."""
    errores = (y_real - y_est) ** 2
    return np.std(errores, ddof=1) / np.sqrt(len(errores))


print(f"MSE test - modelo completo : {mse(y_test, y_pred_completo):.3f}  "
      f"(SE {se_mse(y_test, y_pred_completo):.3f})   [libro: 0.521]")
print(f"MSE test - benchmark media : {mse(y_test, y_pred_benchmark):.3f}  "
      f"(SE {se_mse(y_test, y_pred_benchmark):.3f})   [libro: 1.057]")

# --- Grafico: valor real (y) vs. prediccion (y_pred) --------------------
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharex=True, sharey=True)
paneles = [
    (axes[0], y_pred_completo, f"Modelo completo  (MSE = {mse(y_test, y_pred_completo):.3f})"),
    (axes[1], y_pred_benchmark, f"Benchmark = media  (MSE = {mse(y_test, y_pred_benchmark):.3f})"),
]
for ax, y_hat, titulo in paneles:
    ax.scatter(y_hat, y_test, alpha=0.7, edgecolor="k", linewidth=0.4)
    lim = [min(y_test.min(), y_hat.min()) - 0.3, max(y_test.max(), y_hat.max()) + 0.3]
    ax.plot(lim, lim, "r--", lw=1, label="prediccion perfecta (y = y_pred)")
    ax.set(xlabel="Prediccion  y_pred", title=titulo)
    ax.legend(loc="upper left", fontsize=8)
axes[0].set_ylabel("Valor real  y  (lpsa)")
fig.suptitle("Conjunto de evaluacion: y real vs. prediccion")
fig.tight_layout()
plt.show()


---

### Ejercicio 4 — Prostate Cancer: descenso por gradiente estocástico

Utilizando el dataset de *Prostate Cancer*:
1. Implementar un estimador de cuadrados mínimos con descenso por gradiente estocástico. Utilizar una tasa de aprendizaje $\eta = 0.001$ y un número de iteraciones $iter = 10000$. Graficar la evolución de cada coeficiente $\beta_j$ en función de las iteraciones y, en el mismo gráfico, comparar con los resultados obtenidos previamente o en el libro. **Algunas recomendaciones**: no olvidar escalar los predictores e iniciializar los coeficientes $\beta$ a partir de una distribución normal con media cero y varianza uno.
1. Probar diferentes tasas de aprendizaje y número de iteraciones. ¿Qué ocurre si la tasa de aprendizaje es muy baja? ¿Y si es muy alta? ¿Qué ocurre si el número de iteraciones es muy bajo? ¿Y si es muy alto?
1. Probar diferentes inicializaciones de los coeficientes $\beta_j$. ¿Qué ocurre si se inicializan todos en cero? ¿Y si se inicializan en valores aleatorios?
1. Pensar e investigar cómo se podría mejorar este algoritmo. En particular, considerar condiciones de parada y técnicas por *batch*. No es necesario implementar ninguna de esas mejoras, pero sí pensar en cómo se podrían implementar y qué ventajas tendrían.

In [ ]:
# 4.1 — Estimador de cuadrados minimos por descenso por gradiente estocastico (SGD)

# Reusamos los predictores YA estandarizados de 3.1 y les agregamos la columna de 1s.
X_sgd = np.column_stack([np.ones(len(X_train)), X_train.to_numpy()])
y_sgd = y_train.to_numpy()
n, p = X_sgd.shape                       # n = 67 observaciones ; p = 9 parametros
nombres_coef = ["Intercept"] + predictores

# Solucion exacta de cuadrados minimos (la misma de 3.2 / Tabla 3.2), para comparar.
beta_ols = np.linalg.solve(X_sgd.T @ X_sgd, X_sgd.T @ y_sgd)


def sgd_cuadrados_minimos(X, y, lr, n_iter, beta_inicial, seed=0):
    """Descenso por gradiente estocastico para regresion lineal.

    En cada iteracion toma UNA sola muestra i al azar y da un paso en contra del
    gradiente del error cuadratico de esa muestra:

        error_i   = y_i - x_i . beta
        gradiente = -2 * x_i * error_i          # derivada de (y_i - x_i . beta)^2
        beta      <- beta - lr * gradiente

    Devuelve el historial completo de beta (n_iter + 1 filas) para poder graficar
    su evolucion.
    """
    rng = np.random.default_rng(seed)
    beta = beta_inicial.astype(float).copy()
    historial = np.empty((n_iter + 1, len(beta)))
    historial[0] = beta
    m = len(y)
    for t in range(1, n_iter + 1):
        i = rng.integers(m)                       # indice aleatorio (muestreo con reemplazo)
        error_i = y[i] - X[i] @ beta
        gradiente = -2.0 * X[i] * error_i
        beta = beta - lr * gradiente
        historial[t] = beta
    return historial


# Inicializacion recomendada por el enunciado: normal(0, 1).
beta_0 = np.random.default_rng(42).normal(0.0, 1.0, size=p)

eta = 0.001
n_iter = 10_000
hist = sgd_cuadrados_minimos(X_sgd, y_sgd, lr=eta, n_iter=n_iter, beta_inicial=beta_0, seed=0)

# Comparacion numerica: beta final de SGD vs. solucion exacta
comparacion = pd.DataFrame({
    "SGD (final)": hist[-1],
    "OLS exacto": beta_ols,
}, index=nombres_coef).round(4)
print(comparacion)
print(f"\nDistancia ||beta_SGD - beta_OLS|| = {np.linalg.norm(hist[-1] - beta_ols):.4f}")

# --- Grafico: evolucion de cada beta_j en funcion de las iteraciones -----
# Linea llena = trayectoria de SGD ; linea punteada del mismo color = valor OLS (Tabla 3.2).
fig, ax = plt.subplots(figsize=(9, 5.5))
colores = plt.cm.tab10(np.linspace(0, 1, p))
for j, nombre in enumerate(nombres_coef):
    ax.plot(hist[:, j], color=colores[j], lw=1, label=nombre)
    ax.axhline(beta_ols[j], color=colores[j], ls="--", lw=0.8)
ax.set(xlabel="Iteracion", ylabel="beta_j",
       title="SGD: evolucion de los coeficientes (punteado = solucion exacta OLS / Tabla 3.2)")
ax.legend(ncol=3, fontsize=8, loc="lower right")
fig.tight_layout()
plt.show()

# Nota: con paso (lr) fijo, SGD no converge a un punto: cae rapido hacia la zona de
# la solucion y despues queda "rebotando" en una bola de ruido a su alrededor.


In [ ]:
# 4.2 — Efecto de la tasa de aprendizaje (eta) y del numero de iteraciones

# Para medir "que tan bien va" usamos la distancia entre beta_t y la solucion exacta OLS
# en cada iteracion.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# Panel 1: de una tasa muy baja hasta la recomendada, con n_iter fijo.
for eta_prueba in [1e-5, 1e-4, 1e-3, 1e-2]:
    h = sgd_cuadrados_minimos(X_sgd, y_sgd, lr=eta_prueba, n_iter=n_iter, beta_inicial=beta_0, seed=0)
    axes[0].plot(np.linalg.norm(h - beta_ols, axis=1), lw=1, label=f"eta = {eta_prueba:g}")
axes[0].set(xlabel="Iteracion", ylabel="distancia a la solucion OLS",
            title="Tasa baja  ->  recomendada", yscale="log")
axes[0].legend(fontsize=8)

# Panel 2: una tasa demasiado alta hace que los pasos "sobre-corrijan" y beta diverja.
# (np.errstate silencia los avisos de overflow que produce la divergencia numerica.)
with np.errstate(all="ignore"):
    for eta_prueba in [1e-3, 1e-1]:
        h = sgd_cuadrados_minimos(X_sgd, y_sgd, lr=eta_prueba, n_iter=n_iter, beta_inicial=beta_0, seed=0)
        axes[1].plot(np.linalg.norm(h - beta_ols, axis=1), lw=1, label=f"eta = {eta_prueba:g}")
axes[1].set(xlabel="Iteracion", ylabel="distancia a la solucion OLS",
            title="Tasa muy alta  ->  diverge", yscale="log")
axes[1].legend(fontsize=8)

# Panel 3: con la tasa recomendada, muchas mas iteraciones NO mejoran la precision final.
h_largo = sgd_cuadrados_minimos(X_sgd, y_sgd, lr=eta, n_iter=60_000, beta_inicial=beta_0, seed=0)
axes[2].plot(np.linalg.norm(h_largo - beta_ols, axis=1), lw=1)
axes[2].set(xlabel="Iteracion", ylabel="distancia a la solucion OLS",
            title=f"eta = {eta:g}, hasta 60000 iteraciones", yscale="log")

fig.tight_layout()
plt.show()

print(
    "Tasa de aprendizaje (eta):\n"
    "  - Muy baja (1e-5): cada paso es minusculo; en 10000 iteraciones casi no se\n"
    "    mueve de la inicializacion. Converge, pero necesita muchisimas mas iteraciones.\n"
    "  - Intermedia (1e-3): baja rapido y se estabiliza en una bola de ruido chica\n"
    "    alrededor de la solucion exacta.\n"
    "  - Alta (1e-2): llega mas rapido pero la bola de ruido es mas grande (mas jitter).\n"
    "  - Muy alta (1e-1): los pasos sobre-corrigen, el error crece sin limite -> diverge.\n"
    "\nNumero de iteraciones:\n"
    "  - Muy pocas: se corta antes de llegar a la zona de la solucion -> beta sesgado\n"
    "    hacia beta_0.\n"
    "  - Muchas: una vez dentro de la bola de ruido no mejora mas; solo gasta tiempo.\n"
    "    Para exprimir precision haria falta bajar eta con el tiempo o promediar los\n"
    "    ultimos beta."
)


In [ ]:
# 4.3 — Efecto de la inicializacion de los coeficientes

# Probamos 4 puntos de partida distintos, todo lo demas igual (eta y n_iter fijos).
inicializaciones = {
    "ceros": np.zeros(p),
    "normal(0,1) - semilla A": np.random.default_rng(1).normal(0, 1, p),
    "normal(0,1) - semilla B": np.random.default_rng(2).normal(0, 1, p),
    "lejos (todos = 5)": np.full(p, 5.0),
}

fig, ax = plt.subplots(figsize=(9, 5))
for etiqueta, b0 in inicializaciones.items():
    h = sgd_cuadrados_minimos(X_sgd, y_sgd, lr=eta, n_iter=n_iter, beta_inicial=b0, seed=0)
    ax.plot(np.linalg.norm(h - beta_ols, axis=1), lw=1, label=etiqueta)
ax.set(xlabel="Iteracion", ylabel="distancia a la solucion OLS",
       title="SGD desde distintas inicializaciones", yscale="log")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

print(
    "El problema de cuadrados minimos es convexo: tiene un unico minimo. Por eso todas\n"
    "las inicializaciones terminan en la misma bola de ruido; lo que cambia es el CAMINO\n"
    "y cuantas iteraciones tardan en llegar, no el destino.\n"
    "  - En cero: funciona perfecto. Ademas arranca mas cerca que una normal(0,1),\n"
    "    porque varios coeficientes de la solucion son chicos.\n"
    "  - Aleatoria (normal(0,1)): equivalente a cero salvo unas pocas iteraciones extra\n"
    "    al principio.\n"
    "  - Lejos (todos = 5): solo agrega iteraciones para 'volver'; despues es igual.\n"
    "La recomendacion de romper simetria con valores aleatorios importa en modelos NO\n"
    "convexos (p. ej. redes neuronales); en regresion lineal es indistinto."
)


In [ ]:
# 4.4 — Pensar e investigar cómo se podría mejorar este algoritmo
# TODO


---

### Ejercicio 5 — Código fuente de LinearRegression en scikit-learn

Ir a la documentación y al código de la [regresión lineal en Scikit-Learn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html). Familiarizarse con ella. Leer el comentario acerca de la implementación en "Notes". Luego, ir al código y ver cómo está implementado el `.fit()`. ¿Qué diferencias hay con lo visto?


**Mi respuesta:**

- 


---

### Ejercicio 6 — Solvers de Ridge en scikit-learn

Ir a la documentación de la [regresión Ridge en Scikit-Learn](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html#sklearn.linear_model.Ridge). Leer acerca de los *solvers* disponibles y sus diferencias. ¿Cómo se relacional con lo visto? Volveremos sobre este ejercicio.


**Mi respuesta:**

- 
